# 08D — Baseline fairness, mapped-distance checks, and condition-rule robustness

This follow-up asks:
1. Does a model improve on a historical constant **when both use the same weighting objective**?
2. What does the near-fire dozer group measure, and is its association sensitive to how distance is summarized?
3. Do the condition descriptions survive nearby cutoffs, alternative weights, and small changes in tree depth?

This is an **exploratory follow-up designed after reviewing 08C**. It is not independent confirmation. Every planned comparison is retained; the notebook does not choose the best test-year rule or overwrite the paper.

Save this notebook and **Run All** in Antigravity with the project `.venv` kernel. Sections must run in order. No downloads, GPU, or large boosting-model retraining are required. Nine small trees are fitted. A fresh timestamped output folder is created; automatic resume is not implemented.

Existing NIFC labels, fireline units, prediction times, and model predictions remain unchanged. Relative humidity is forecast atmospheric humidity, not measured vegetation moisture. Original source-record length is not complete constructed-fireline length. No optimal width, causal construction effect, or tactical instruction is estimated.


## 0. Configuration
The completed 08C run and its inputs are pinned. The full run uses 2,000 resamples of complete fires. Small verification runs are marked as such.


In [ ]:
import os  # Configurable project root for the public release.
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version as package_version
import json, hashlib, sys, platform, uuid, zlib, time, importlib.util
required=["numpy","pandas","pyarrow","scikit-learn","matplotlib","IPython"]
for module in ["numpy","pandas","pyarrow","sklearn","matplotlib","IPython"]:
    if importlib.util.find_spec(module) is None:
        raise ImportError("Use the project .venv kernel; missing package: "+module)
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import roc_auc_score
from IPython.display import display, Markdown

def find_root():
    for p in [Path.cwd(),*Path.cwd().parents,Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))]:
        if (p/"data/processed/modeling_table_assembly_v1/fireline_hold_modeling_master_v1.parquet").is_file():
            return p.resolve()
    raise FileNotFoundError("Open the notebook in the project or set PROJECT_ROOT.")
PROJECT_ROOT=find_root()

C_RUN_NAME="08C_20260930T234354_724489Z_3a31ea"
B_RUN_NAME="08B_20260930T224636_898345Z_38d03c"
C_DIR=PROJECT_ROOT/"outputs/08_condition_outcomes"/C_RUN_NAME
B_DIR=PROJECT_ROOT/"outputs/08_condition_outcomes"/B_RUN_NAME
UNIT_PATH=B_DIR/"08B_existing_units_with_diagnostic_weights.parquet"
PARENT_PATH=B_DIR/"08B_retained_source_record_summaries.parquet"
MASTER_PATH=PROJECT_ROOT/"data/processed/modeling_table_assembly_v1/fireline_hold_modeling_master_v1.parquet"
OOF_PATH=PROJECT_ROOT/"data/processed/baseline_modeling_v1/oof_predictions/E_FULL_EXTENDED__row.parquet"
FORWARD_PATH=PROJECT_ROOT/"data/processed/forward_temporal_generalization_v1/FORWARD_TEMPORAL_PREDICTIONS_V1.parquet"
NOTEBOOK_PATH=PROJECT_ROOT/"notebooks/08D_baseline_and_condition_robustness.ipynb"
BOOTSTRAP_REPLICATES=2000
IS_SMOKE_TEST=False
RANDOM_SEED=20261001  # matches 08C for the depth-3 reproduction
MIN_LEAF_UNITS=500; MIN_LEAF_PARENTS=100; MIN_LEAF_FIRES=10
MIN_LEAF_WEIGHT_FRACTION=.01
MIN_TEST_UNITS=200; MIN_TEST_PARENTS=50; MIN_TEST_FIRES=10
MIN_CONTRAST_PARENTS_PER_FIRE=5
WIDTH_REVIEW_FT=300.0
BUILD_ID="08D_V1_BASELINE_AND_CONDITION_ROBUSTNESS"
RUN_ID=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")+"_"+uuid.uuid4().hex[:6]
RUN_DIR=PROJECT_ROOT/"outputs/08_condition_outcomes"/("08D_"+RUN_ID)
MODEL_DIR=RUN_DIR/"models"; FIG_DIR=RUN_DIR/"figures"
WEIGHT_MODES=["unit","source","fire_unit","fire_source"]
WEIGHT_LABELS={"unit":"Equal fireline units","source":"Equal source records",
 "fire_unit":"Equal fires; equal units within fire","fire_source":"Equal fires; equal sources within fire"}
TREE_SPECS=[{"name":f"depth_{depth}__{mode}","depth":depth,"weight_mode":mode}
            for depth in [2,3,4] for mode in ["fire_unit","fire_source","source"]]
PREDICTORS={
 "humidity":("hrrr_window_0_18h_rh2_pct_mean","Mean forecast relative humidity, 0–18 h","%",(0,100)),
 "gust":("hrrr_gust_mps_p18h","Forecast gust at +18 h","m/s",(0,None)),
 "wind":("hrrr_window_0_18h_wind10_speed_mean_mps","Mean forecast wind speed, 0–18 h","m/s",(0,None)),
 "temperature":("hrrr_window_0_18h_t2_c_max","Maximum forecast temperature, 0–18 h","deg C",(None,None)),
 "slope":("terrain_slope_mean_deg","Mean terrain slope","degrees",(0,90)),
 "elevation":("terrain_elevation_mean_m","Mean elevation","m",(None,None)),
 "crown_depth":("landfire_line_crown_depth_mean","Mean crown depth sampled on the fireline","m",(0,None)),
 "canopy_cover":("landfire_line_cc_mean","Mean canopy cover sampled on the fireline","%",(0,100)),
 "distance":("local_fire_to_line_distance_geom_m","Closest mapped fire-area-to-fireline distance","m",(0,None)),
 "width":("LineWidthFeet","Reported fireline width","m",(0,WIDTH_REVIEW_FT*0.3048)),
}

ENVIRONMENT_KEYS=[k for k in PREDICTORS if k!="width"]
def saved_notebook_source_hash():
    doc=json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
    return hashlib.sha256(json.dumps([c["source"] for c in doc["cells"]],ensure_ascii=False).encode()).hexdigest()
NOTEBOOK_SOURCE_HASH_AT_START=saved_notebook_source_hash()
print("08D output:",RUN_DIR)


## 1. Load and verify the frozen population
The same 304,143 fireline units and 381 fires are retained. Saved predictions are joined by verified identifiers, never by assumed row order. Mapping diagnostics are not used as new predictors or as outcome-based exclusions.

**Distance definitions matter:** the original geometric predictor is the minimum distance between the entire mapped fireline-unit geometry and the NIROPS fire polygon at the assigned time. Any intersection gives zero. The exact-distance summaries instead describe sampled positions along the fireline relative to the polygon boundary. Differences between these summaries need not be map errors.


In [ ]:
def require(ok,message):
    if not bool(ok): raise AssertionError(message)

def safe_json(x):
    if isinstance(x,dict): return {str(k):safe_json(v) for k,v in x.items()}
    if isinstance(x,(list,tuple)): return [safe_json(v) for v in x]
    if isinstance(x,np.ndarray): return safe_json(x.tolist())
    if isinstance(x,Path): return str(x)
    if isinstance(x,(np.integer,)): return int(x)
    if isinstance(x,(np.bool_,)): return bool(x)
    if isinstance(x,(float,np.floating)):
        if not np.isfinite(x): return None
        return float(x)
    return x

def write_json(path,obj):
    Path(path).write_text(json.dumps(safe_json(obj),indent=2,allow_nan=False),encoding="utf-8")

def digest(frame,columns,keys):
    f=frame[columns].sort_values(keys).reset_index(drop=True)
    return hashlib.sha256(pd.util.hash_pandas_object(f,index=False).to_numpy().tobytes()).hexdigest()


required_paths=[UNIT_PATH,PARENT_PATH,MASTER_PATH,OOF_PATH,FORWARD_PATH,C_DIR/"08C_run_manifest.json",
 C_DIR/"08C_quality_checks.json",C_DIR/"08C_unit_lookup.parquet",C_DIR/"08C_out_of_sample_predictions.parquet",
 C_DIR/"08C_frozen_rule_predictions.parquet",C_DIR/"frozen_rule_cards.csv",C_DIR/"models/condition_tree__frozen_2020_2022/model.json"]
for path in required_paths: require(path.is_file(),"Missing required input: "+str(path))
manifest=json.loads((C_DIR/"08C_run_manifest.json").read_text(encoding="utf-8"))
quality=json.loads((C_DIR/"08C_quality_checks.json").read_text(encoding="utf-8"))
require(not manifest["is_smoke_test"] and manifest["bootstrap_replicates"]==2000,"A completed production 08C run is required.")
for key in ["frozen_population_preserved","fire_and_parent_disjointness_checked","out_of_sample_predictions_unique_and_complete"]:
    require(quality[key],"08C quality check failed: "+key)
require(manifest["n_refitted_trees"]==72 and manifest["n_frozen_trees"]==3,"Incomplete 08C run.")
input_stats={str(p):{"bytes":p.stat().st_size,"modified_ns":p.stat().st_mtime_ns} for p in required_paths}
for path,stats in input_stats.items():
    if path in manifest["input_files"]:
        require(stats==manifest["input_files"][path],"Input differs from the completed 08C snapshot: "+path)
units=pd.read_parquet(UNIT_PATH).sort_values("unit_uid").reset_index(drop=True)
require(len(units)==304143 and units["_irwin_norm"].nunique()==381,"Population changed.")
require(units["unit_uid"].is_unique and units["unit_uid"].notna().all(),"Invalid unit IDs.")
units["row_id"]=np.arange(len(units),dtype=np.int64)
lookup=pd.read_parquet(C_DIR/"08C_unit_lookup.parquet").sort_values("row_id").reset_index(drop=True)
identity=["row_id","unit_uid","_irwin_norm","source_record_key","Year","outer_fold","y","prediction_anchor_utc"]
pd.testing.assert_frame_equal(units[identity],lookup[identity],check_dtype=False)
for col in ["Year","outer_fold"]:
    require(units.groupby("_irwin_norm")[col].nunique().eq(1).all(),"A fire spans "+col)
for col in ["_irwin_norm","Year","outer_fold","y"]:
    require(units.groupby("source_record_key")[col].nunique().eq(1).all(),"Parent inconsistency: "+col)
identity_hash=digest(units,identity,["unit_uid"])
parent=pd.read_parquet(PARENT_PATH,columns=["source_record_key","n_prediction_times","retained_length_ratio"])
units=units.merge(parent,on="source_record_key",how="left",validate="many_to_one")
require(units["n_prediction_times"].notna().all(),"Missing parent join.")
diagnostic_columns=["operational_anchor_clean","prediction_t0_any_inside","prediction_anchor_idx",
 "exact_t0_exact_mean_distance_m","exact_t0_exact_median_distance_m","exact_t0_exact_p10_distance_m",
 "exact_t0_exact_p90_distance_m","exact_t0_exact_min_distance_m","exact_t0_cached_inside_fraction",
 "exact_t0_sample_points","exact_t0_observation_index","exact_history_any_cached_inside"]
schema=set(pq.read_schema(MASTER_PATH).names)
require(set(diagnostic_columns).issubset(schema),"Master mapping-diagnostic schema changed.")
master=pd.read_parquet(MASTER_PATH,columns=["unit_uid","hold_label","prediction_anchor_utc"]+diagnostic_columns)
check=units[["unit_uid","y","prediction_anchor_utc"]].merge(master,on="unit_uid",validate="one_to_one",suffixes=("_unit","_master"))
require(check["y"].eq(check["hold_label"]).all(),"Master labels changed.")
require(check["prediction_anchor_utc_unit"].eq(check["prediction_anchor_utc_master"]).all(),"Master times changed.")
units=units.merge(master[["unit_uid"]+diagnostic_columns],on="unit_uid",how="left",validate="one_to_one")
for key,(source,label,measurement,bounds) in PREDICTORS.items():
    x=pd.to_numeric(units[source],errors="coerce")
    if key=="width": x=x*.3048
    lo,hi=bounds; valid=np.isfinite(x)
    if lo is not None: valid &= x.ge(lo)
    if hi is not None: valid &= x.le(hi)
    if key=="width": valid &= x.gt(0)
    units["x_"+key]=x.where(valid)
units["type_value"]=units["fireline_type"].fillna("__MISSING__").astype(str)
frozen_payload=json.loads((C_DIR/"models/condition_tree__frozen_2020_2022/model.json").read_text(encoding="utf-8"))
frozen_paths={int(k):v for k,v in frozen_payload["paths"].items()}
frozen_probs={int(k):v for k,v in frozen_payload["leaf_probabilities"].items()}
saved_predictions=pd.read_parquet(C_DIR/"08C_out_of_sample_predictions.parquet")
saved_predictions=saved_predictions.loc[saved_predictions["variant"].eq("condition_tree")].copy()
require(not saved_predictions.duplicated(["scheme","row_id"]).any(),"Duplicate 08C predictions.")
RUN_DIR.mkdir(parents=True,exist_ok=False); MODEL_DIR.mkdir(); FIG_DIR.mkdir()
tables={}; phase_done={"load"}
print(f"Verified {len(units):,} units / {units['_irwin_norm'].nunique()} fires.")


## 2. Weighting and whole-fire uncertainty
Four choices are crossed deliberately:
- **Unit:** every fireline unit has equal influence.
- **Source:** each original source record has equal influence, divided among its fireline units.
- **Fire/unit:** each fire has equal influence, divided among its fireline units.
- **Fire/source:** each fire has equal influence, then each source record within that fire has equal influence.

Each historical constant is a training-only weighted Held fraction. Every model is scored against every constant on identical test observations. Matching the baseline and evaluation weights is the main comparison; crossed choices show sensitivity.

Bootstrap draws resample complete fires. Pooled summaries use ratios of resampled weighted sums, not averages of fire means. Paired differences use identical draws. Combined-year draws allow year composition to vary. Intervals hold predictions fixed and do not include model-selection or retraining uncertainty.

For log loss, probabilities are clipped to [10^-7, 1 - 10^-7] to keep the score finite. All saved intervals are pointwise; there is no correction for selecting findings after examining many comparisons.


In [ ]:
def need(*names):
    missing=[s for s in names if s not in phase_done]
    require(not missing,"Run the preceding notebook sections first: "+", ".join(missing))

def source_weights(frame):
    require(len(frame)>0,"Cannot weight an empty population.")
    counts=frame.groupby("source_record_key")["source_record_key"].transform("size")
    w=1.0/counts.to_numpy(dtype=float)
    totals=pd.Series(w,index=frame.index).groupby(frame["source_record_key"]).sum()
    require(np.allclose(totals,1.0),"Source weights do not sum to one.")
    return w


def training_weights(frame,mode="fire_source"):
    if mode=="unit": return np.ones(len(frame))
    if mode=="fire_unit":
        return 1/frame.groupby("_irwin_norm")["_irwin_norm"].transform("size").to_numpy(dtype=float)
    w=source_weights(frame)
    if mode=="source": return w
    require(mode=="fire_source","Unknown weight mode: "+mode)
    parents=frame[["_irwin_norm","source_record_key"]].drop_duplicates().groupby("_irwin_norm").size()
    return w/frame["_irwin_norm"].map(parents).to_numpy(dtype=float)

def clustered_parts(frame,values,weights):
    f=pd.DataFrame({"fire":frame["_irwin_norm"].to_numpy(),"num":np.asarray(values)*weights,"den":weights})
    return f.groupby("fire",sort=True)[["num","den"]].sum()

def resampled_ratios(parts,key):
    require(len(parts)>0 and (parts["den"]>0).all(),"Empty/nonpositive aggregate.")
    num=parts["num"].to_numpy(); den=parts["den"].to_numpy()
    if len(parts)<2: return np.full(BOOTSTRAP_REPLICATES,np.nan)
    # Same key and ordered fires give paired draws across candidate predictions.
    seed=(RANDOM_SEED+zlib.crc32((key+"|"+"|".join(parts.index.astype(str))).encode()))%(2**32)
    rng=np.random.default_rng(seed); draws=np.empty(BOOTSTRAP_REPLICATES)
    for start in range(0,BOOTSTRAP_REPLICATES,128):
        stop=min(start+128,BOOTSTRAP_REPLICATES)
        ids=rng.integers(0,len(parts),(stop-start,len(parts)))
        draws[start:stop]=num[ids].sum(axis=1)/den[ids].sum(axis=1)
    return draws

def interval(draws):
    finite=np.asarray(draws)[np.isfinite(draws)]
    return tuple(np.quantile(finite,[.025,.975])) if len(finite)>=2 else (np.nan,np.nan)

def stat_from_parts(parts,key):
    if len(parts)==0: return np.nan,np.nan,np.nan
    point=parts["num"].sum()/parts["den"].sum()
    lo,hi=interval(resampled_ratios(parts,key))
    return float(point),lo,hi

def error_comparisons(frame,prediction_dict,models,baselines,scope,scheme):
    rows=[]; y=frame["y"].to_numpy(dtype=float)
    for mode in WEIGHT_MODES:
        w=training_weights(frame,mode)
        for metric in ["brier","log_loss"]:
            summaries={}
            for name,p in prediction_dict.items():
                p=np.asarray(p,dtype=float)
                require(len(p)==len(frame) and np.isfinite(p).all() and ((p>=0)&(p<=1)).all(),"Invalid prediction "+name)
                q=np.clip(p,1e-7,1-1e-7)
                loss=(p-y)**2 if metric=="brier" else -(y*np.log(q)+(1-y)*np.log(1-q))
                parts=clustered_parts(frame,loss,w)
                point=float(parts["num"].sum()/parts["den"].sum())
                draws=resampled_ratios(parts,scheme+"|"+scope+"|"+mode)
                summaries[name]=(point,draws)
            for model in models:
                point,draws=summaries[model]
                for baseline in baselines:
                    bp,bd=summaries[baseline]; lo,hi=interval(draws-bd)
                    rows.append({"scheme":scheme,"scope":scope,"metric":metric,"model":model,
                     "evaluation_weight":mode,"baseline_weight":baseline.removeprefix("prior_"),
                     "model_error":point,"baseline_error":bp,"difference":point-bp,
                     "ci_lower":lo,"ci_upper":hi,"n_units":len(frame),"n_source_records":frame["source_record_key"].nunique(),
                     "n_fires":frame["_irwin_norm"].nunique(),"matched_weighting":baseline=="prior_"+mode})
    return rows

def group_summary(frame,mask,name,scope,mode,comparison_family="frozen"):
    mask=np.asarray(mask,dtype=bool); require(len(mask)==len(frame),"Mask length mismatch.")
    basic={"group":name,"scope":scope,"weight":mode,"family":comparison_family,"n_units":int(mask.sum()),
     "n_source_records":frame.loc[mask,"source_record_key"].nunique(),"n_fires":frame.loc[mask,"_irwin_norm"].nunique(),
     "n_held":int(frame.loc[mask,"y"].sum()),"n_burned_over":int((1-frame.loc[mask,"y"]).sum())}
    w=source_weights(frame) if mode in ["source","fire_source"] else np.ones(len(frame))
    g=clustered_parts(frame.loc[mask],frame.loc[mask,"y"],w[mask])
    if mode.startswith("fire_") and len(g): g=pd.DataFrame({"num":g["num"]/g["den"],"den":1.0})
    point,lo,hi=stat_from_parts(g,"group|"+scope)
    basic.update(held_fraction=point,ci_lower=lo,ci_upper=hi,
      degenerate_interval=bool(len(g)>1 and np.ptp(g["num"]/g["den"])==0),
      support_pass=basic["n_units"]>=MIN_TEST_UNITS and basic["n_source_records"]>=MIN_TEST_PARENTS and basic["n_fires"]>=MIN_TEST_FIRES)
    # Within-fire contrasts retain source shares allocated over the full scope.
    a=clustered_parts(frame.loc[mask],frame.loc[mask,"y"],w[mask])
    b=clustered_parts(frame.loc[~mask],frame.loc[~mask,"y"],w[~mask])
    ca=frame.loc[mask].groupby("_irwin_norm")["source_record_key"].nunique()
    cb=frame.loc[~mask].groupby("_irwin_norm")["source_record_key"].nunique()
    eligible=ca[ca.ge(MIN_CONTRAST_PARENTS_PER_FIRE)].index.intersection(cb[cb.ge(MIN_CONTRAST_PARENTS_PER_FIRE)].index)
    diff=(a["num"]/a["den"]-b["num"]/b["den"]).reindex(eligible).dropna().sort_index()
    parts=pd.DataFrame({"num":diff,"den":1.0})
    delta,dlo,dhi=stat_from_parts(parts,"contrast|"+scope)
    loo=(diff.sum()-diff)/(len(diff)-1) if len(diff)>1 else pd.Series(dtype=float)
    basic.update(contrast_fires=len(diff),within_fire_difference=delta,contrast_ci_lower=dlo,contrast_ci_upper=dhi,
                 leave_one_fire_out_min=float(loo.min()) if len(loo) else np.nan,
                 leave_one_fire_out_max=float(loo.max()) if len(loo) else np.nan)
    return basic,diff

need("load")
# Synthetic checks expose weighting and bootstrap aggregation mistakes.
toy=pd.DataFrame({"_irwin_norm":["a"]*10+["b"]*2,"source_record_key":["a1"]*9+["a2"]+["b1"]*2})
require(np.allclose(pd.Series(training_weights(toy,"fire_source")).groupby(toy["_irwin_norm"]).sum(),1),"Fire/source weighting failed.")
require(np.allclose(pd.Series(source_weights(toy)).groupby(toy["source_record_key"]).sum(),1),"Parent weighting failed.")
parts=pd.DataFrame({"num":[0.,100.],"den":[1.,100.]},index=["a","b"])
require(np.isclose(stat_from_parts(parts,"toy")[0],100/101),"Pooled ratio accidentally became a fire average.")
require(np.array_equal(resampled_ratios(parts,"same"),resampled_ratios(parts,"same")),"Paired draws differ.")
empty=group_summary(units.iloc[:10],np.zeros(10,dtype=bool),"empty","toy","fire_source")[0]
require(empty["n_units"]==0 and np.isnan(empty["held_fraction"]),"Empty groups not handled.")
phase_done.add("statistics")
print("Weighting, paired resampling, and empty-group safeguards passed.")


## 3. Recompute four training-only baselines and compare saved predictions
No LightGBM model is refitted. Each five-fold prior excludes the entire test fold; each forward-year prior uses strictly preceding years. Baselines are not estimated from test-year outcomes.

The combined five-fold population includes 2020, whereas forward-year tests start in 2021. A matched 2021–2024 five-fold summary is also saved so these populations are not confused. The full four-by-four weighting table separates a baseline change from a scoring change.


In [ ]:
need("statistics")
baseline_rows=[]; baseline_predictions=[]; evaluation_rows=[]; prediction_exports=[]
for scheme in ["five_fold","forward_year"]:
    saved=saved_predictions.loc[saved_predictions["scheme"].eq(scheme)].sort_values("row_id").reset_index(drop=True)
    expected=units if scheme=="five_fold" else units.loc[units["Year"].ge(2021)]
    require(set(saved["row_id"])==set(expected["row_id"]),"Incomplete saved predictions.")
    frame=units.set_index("row_id",drop=False).loc[saved["row_id"]].reset_index(drop=True)
    path,column=(OOF_PATH,"pred_prob") if scheme=="five_fold" else (FORWARD_PATH,"pred_forward_fixed_05c")
    ref=pd.read_parquet(path)
    require(ref["unit_uid"].is_unique,"Duplicate LightGBM reference IDs.")
    joined=frame[["unit_uid","y","_irwin_norm","Year","outer_fold"]].merge(
      ref[["unit_uid","hold_label","_irwin_norm","Year","outer_fold",column]+
          (["pred_historical_prevalence_prior"] if scheme=="forward_year" else [])],
      on="unit_uid",how="left",validate="one_to_one",suffixes=("_frame","_ref"))
    require(joined["y"].eq(joined["hold_label"]).all(),"LightGBM labels differ.")
    for field in ["_irwin_norm","Year","outer_fold"]:
        require(joined[field+"_frame"].eq(joined[field+"_ref"]).all(),"LightGBM split identity differs: "+field)
    predictions={"condition_tree":saved["p_held"].to_numpy(),"LightGBM":joined[column].to_numpy()}
    for mode in WEIGHT_MODES: predictions["prior_"+mode]=np.full(len(frame),np.nan)
    split_column="outer_fold" if scheme=="five_fold" else "Year"
    for split in sorted(frame[split_column].unique()):
        train=units.loc[units["outer_fold"].ne(split)] if scheme=="five_fold" else units.loc[units["Year"].lt(split)]
        mask=frame[split_column].eq(split).to_numpy(); test=frame.loc[mask]
        require(not(set(train["_irwin_norm"])&set(test["_irwin_norm"])),"Baseline fire leakage.")
        require(not(set(train["source_record_key"])&set(test["source_record_key"])),"Baseline source leakage.")
        for mode in WEIGHT_MODES:
            prior=float(np.average(train["y"],weights=training_weights(train,mode)))
            predictions["prior_"+mode][mask]=prior
            baseline_rows.append({"scheme":scheme,"split":int(split),"weight":mode,"held_probability":prior,
              "train_years":",".join(map(str,sorted(train["Year"].unique()))),"train_units":len(train),
              "train_sources":train["source_record_key"].nunique(),"train_fires":train["_irwin_norm"].nunique(),
              "test_units":len(test),"test_fires":test["_irwin_norm"].nunique()})
    require(np.allclose(predictions["prior_fire_source"],saved["baseline_p"],rtol=0,atol=1e-12),"08C training prior not reproduced.")
    if scheme=="forward_year":
        require(np.allclose(predictions["prior_unit"],joined["pred_historical_prevalence_prior"],rtol=0,atol=1e-12),
                "Original manuscript historical prior not reproduced.")
    export=frame[identity].copy()
    for name,p in predictions.items(): export[name]=p
    export["scheme"]=scheme; prediction_exports.append(export)
    scopes=[("all_test_years",np.ones(len(frame),dtype=bool))]
    scopes += [(str(year),frame["Year"].eq(year).to_numpy()) for year in sorted(frame["Year"].unique())]
    if scheme=="five_fold": scopes.append(("matched_2021_2024",frame["Year"].ge(2021).to_numpy()))
    for scope,mask in scopes:
        evaluation_rows.extend(error_comparisons(frame.loc[mask],{k:v[mask] for k,v in predictions.items()},
          ["condition_tree","LightGBM"],["prior_"+m for m in WEIGHT_MODES],scope,scheme))
        print(scheme,scope,"scored.",flush=True)
    print(scheme,"baseline and error comparisons complete.",flush=True)
tables["training_baselines"]=pd.DataFrame(baseline_rows)
tables["baseline_error_comparisons"]=pd.DataFrame(evaluation_rows)
pd.concat(prediction_exports,ignore_index=True).to_parquet(RUN_DIR/"08D_matched_predictions_and_baselines.parquet",index=False)
tables["training_baselines"].to_csv(RUN_DIR/"training_baselines.csv",index=False)
tables["baseline_error_comparisons"].to_csv(RUN_DIR/"baseline_error_comparisons.csv",index=False)
phase_done.add("baselines")
display(tables["baseline_error_comparisons"].query("scope == 'all_test_years' and metric == 'brier' and matched_weighting"))


## 4. Replay the eight frozen 08C groups exactly
All eight paths are retained. Displayed rounded cutoffs are not used for membership; float32 predictor values are compared against full-precision saved thresholds.

For group rates, source weights are allocated over the full test scope before selecting a group. Each original source contributes the fraction of its fireline units in that group. Equal-fire summaries then give each represented fire equal influence. The within-fire comparison uses fires with at least five represented source records on each side.

Intervals are pointwise exploratory intervals, not simultaneous guarantees across the many checks. Leave-one-fire-out contrasts expose dependence on individual fires.


In [ ]:
def weighted_median(values,weights):
    values=np.asarray(values,dtype=float);weights=np.asarray(weights,dtype=float)
    keep=np.isfinite(values)&np.isfinite(weights)&(weights>0)
    if not keep.any(): return 0.0
    order=np.argsort(values[keep],kind="stable")
    x=values[keep][order];w=weights[keep][order]
    return float(x[np.searchsorted(np.cumsum(w),w.sum()/2,side="left")])

def fit_preprocessor(train,keys,use_type,weights):
    prep={"keys":list(keys),"use_type":bool(use_type),"medians":{},"all_missing_training":{},
          "type_categories":[],"features":[],"input_dtype":"float32"}
    for key in keys:
        values=train["x_"+key].to_numpy(dtype=float)
        prep["medians"][key]=weighted_median(values,weights)
        prep["all_missing_training"][key]=bool(~np.isfinite(values).any())
        prep["features"].append({"name":key,"kind":"numeric","key":key,
                                 "label":PREDICTORS[key][1],"unit":PREDICTORS[key][2]})
        prep["features"].append({"name":key+"__missing","kind":"missing","key":key,
                                 "label":PREDICTORS[key][1]+" missing","unit":"indicator"})
    if use_type:
        cats=sorted(set(train["type_value"]) - {"__MISSING__","__UNSEEN__"})
        prep["type_categories"]=cats+["__MISSING__","__UNSEEN__"]
        for cat in prep["type_categories"]:
            prep["features"].append({"name":"type="+cat,"kind":"category","category":cat,
                                     "label":"Fireline type = "+cat,"unit":"indicator"})
    return prep

def transform(frame,prep):
    columns=[]; missing=[]
    for key in prep["keys"]:
        values=frame["x_"+key].to_numpy(dtype=float)
        bad=~np.isfinite(values)
        columns += [np.where(bad,prep["medians"][key],values),bad.astype(float)]
        missing.append(bad)
    unseen=np.zeros(len(frame),dtype=bool)
    if prep["use_type"]:
        raw=frame["type_value"].to_numpy(dtype=str)
        unseen=~np.isin(raw,prep["type_categories"])
        values=np.where(unseen,"__UNSEEN__",raw)
        columns += [(values==cat).astype(float) for cat in prep["type_categories"]]
    X=np.column_stack(columns).astype(np.float32)
    require(np.isfinite(X).all(),"Nonfinite transformed predictors.")
    return X,{"any_numeric_missing":np.column_stack(missing).any(axis=1),
              "unseen_type":unseen}

def support_counts(frame):
    return {"n_units":len(frame),"n_source_records":frame["source_record_key"].nunique(),
            "n_fires":frame["_irwin_norm"].nunique()}

def sufficient_training_support(frame,min_units,min_parents,min_fires):
    c=support_counts(frame)
    return c["n_units"]>=min_units and c["n_source_records"]>=min_parents and c["n_fires"]>=min_fires

def extract_paths(clf):
    paths={}
    def walk(node,path):
        left=int(clf.tree_.children_left[node]);right=int(clf.tree_.children_right[node])
        if left<0:
            paths[node]=path;return
        feature=int(clf.tree_.feature[node]);threshold=float(clf.tree_.threshold[node])
        walk(left,path+[{"feature_index":feature,"operator":"<=","threshold":threshold}])
        walk(right,path+[{"feature_index":feature,"operator":">","threshold":threshold}])
    walk(0,[])
    return paths

def describe_path(path,prep):
    result=[]
    for step in path:
        meta=prep["features"][step["feature_index"]]
        goes_right=step["operator"]==">"
        if meta["kind"]=="category":
            result.append("Fireline type "+("is " if goes_right else "is not ")+meta["category"])
        elif meta["kind"]=="missing":
            result.append(meta["label"]+(" = yes" if goes_right else " = no"))
        else:
            threshold=step["threshold"]
            result.append(meta["label"]+" "+step["operator"]+" "+format(threshold,".4g")+" "+meta["unit"])
    return " AND ".join(result) if result else "All eligible fireline units (no supported split)"

def replay_paths(X,paths):
    assigned=np.full(len(X),-999,dtype=int);n_assign=np.zeros(len(X),dtype=int)
    for node,path in paths.items():
        mask=np.ones(len(X),dtype=bool)
        for step in path:
            # sklearn compares float32 inputs against double-precision thresholds.
            # Promote explicitly: NumPy weak Python scalars must not round the threshold.
            col=X[:,step["feature_index"]].astype(np.float64)
            mask &= col<=step["threshold"] if step["operator"]=="<=" else col>step["threshold"]
        assigned[mask]=int(node);n_assign[mask]+=1
    require((n_assign==1).all(),"Rules overlap or leave observations unassigned.")
    return assigned

def model_fingerprint(payload):
    return hashlib.sha256(json.dumps(safe_json(payload),sort_keys=True,separators=(",",":")).encode()).hexdigest()

def fit_rule_tree(train,spec,model_id,min_units=None,min_parents=None,min_fires=None,max_depth=3):
    min_units=MIN_LEAF_UNITS if min_units is None else min_units
    min_parents=MIN_LEAF_PARENTS if min_parents is None else min_parents
    min_fires=MIN_LEAF_FIRES if min_fires is None else min_fires
    require(len(train)>0 and train["y"].isin([0,1]).all(),"Invalid training outcomes.")
    weights=training_weights(train,spec["weight_mode"])
    prep=fit_preprocessor(train,spec["keys"],spec["use_type"],weights)
    X,_=transform(train,prep)
    clf=DecisionTreeClassifier(max_depth=max_depth,min_samples_leaf=min_units,
                              min_weight_fraction_leaf=MIN_LEAF_WEIGHT_FRACTION,
                              random_state=RANDOM_SEED,class_weight=None,criterion="gini")
    clf.fit(X,train["y"].to_numpy(),sample_weight=weights)
    pruned=[]
    def prune(node,idx):
        left=int(clf.tree_.children_left[node]);right=int(clf.tree_.children_right[node])
        if left<0: return
        feature=int(clf.tree_.feature[node]);threshold=clf.tree_.threshold[node]
        test=X[idx,feature].astype(np.float64)<=np.float64(threshold)
        li=idx[test];ri=idx[~test]
        if not(sufficient_training_support(train.iloc[li],min_units,min_parents,min_fires) and
               sufficient_training_support(train.iloc[ri],min_units,min_parents,min_fires)):
            pruned.append(int(node))
            clf.tree_.children_left[node]=-1;clf.tree_.children_right[node]=-1
            clf.tree_.feature[node]=-2;clf.tree_.threshold[node]=-2
        else:
            prune(left,li);prune(right,ri)
    prune(0,np.arange(len(train)))
    leaf=clf.apply(X)
    paths=extract_paths(clf)
    require(np.array_equal(leaf,replay_paths(X,paths)),"Training path replay mismatch.")
    leaf_probs={};train_support={}
    for node in paths:
        mask=leaf==node
        leaf_probs[node]=float(np.average(train.loc[mask,"y"],weights=weights[mask]))
        train_support[node]={**support_counts(train.loc[mask]),"training_held_probability":leaf_probs[node],
                             "support_pass":sufficient_training_support(train.loc[mask],min_units,min_parents,min_fires)}
    baseline=float(np.average(train["y"],weights=weights))
    payload={"model_id":model_id,"spec":spec,"preprocessor":prep,
             "training_years":sorted(train["Year"].unique().tolist()),"training_fires":sorted(train["_irwin_norm"].unique()),
             "training_unit_hash":digest(train,["unit_uid","y"],["unit_uid"]),
             "training_parent_hash":hashlib.sha256("|".join(sorted(train["source_record_key"].unique())).encode()).hexdigest(),
             "training_support":support_counts(train),"baseline_held_probability":baseline,
             "paths":paths,"leaf_probabilities":leaf_probs,"leaf_training_support":train_support,
             "pruned_nodes":pruned,"max_depth":max_depth,
             "support_thresholds":{"units":min_units,"source_records":min_parents,"fires":min_fires},
             "tree_children_left":clf.tree_.children_left.tolist(),"tree_children_right":clf.tree_.children_right.tolist(),
             "tree_feature":clf.tree_.feature.tolist(),"tree_threshold":clf.tree_.threshold.tolist(),
             "tree_classes":clf.classes_.tolist(),"tree_value":clf.tree_.value.tolist()}
    return {"clf":clf,"prep":prep,"paths":paths,"leaf_probs":leaf_probs,"baseline":baseline,
            "payload":payload,"fingerprint":model_fingerprint(payload)}

def predict_rule_tree(bundle,frame):
    X,flags=transform(frame,bundle["prep"])
    leaf=bundle["clf"].apply(X)
    require(np.array_equal(leaf,replay_paths(X,bundle["paths"])),"Test path replay mismatch.")
    p=np.array([bundle["leaf_probs"][int(node)] for node in leaf])
    return p,leaf,flags

def assert_split(train,test,chronological=False):
    require(len(train)>0 and len(test)>0,"Empty train/test population.")
    require(not(set(train["_irwin_norm"])&set(test["_irwin_norm"])),"Fire leakage.")
    require(not(set(train["source_record_key"])&set(test["source_record_key"])),"Source-record leakage.")
    require(not(set(train["unit_uid"])&set(test["unit_uid"])),"Fireline-unit leakage.")
    if chronological:
        require(int(train["Year"].max())<int(test["Year"].min()),"Future/test-year training data.")


need("baselines")
test=units.loc[units["Year"].isin([2023,2024])].copy().reset_index(drop=True)
train=units.loc[units["Year"].isin([2020,2021,2022])].copy().reset_index(drop=True)
assert_split(train,test,chronological=True)
frozen_X,frozen_flags=transform(test,frozen_payload["preprocessor"])
frozen_leaf=replay_paths(frozen_X,frozen_paths)
frozen_p=np.array([frozen_probs[int(leaf)] for leaf in frozen_leaf])
saved_frozen=pd.read_parquet(C_DIR/"08C_frozen_rule_predictions.parquet")
saved_frozen=saved_frozen.loc[saved_frozen["variant"].eq("condition_tree")].set_index("row_id").reindex(test["row_id"])
require(np.array_equal(saved_frozen["leaf_id"],frozen_leaf),"Saved frozen membership does not replay.")
require(np.array_equal(saved_frozen["p_held"],frozen_p),"Saved frozen probabilities changed.")
SCOPES=[("2023_2024",np.ones(len(test),dtype=bool))]+[(str(y),test["Year"].eq(y).to_numpy()) for y in [2023,2024]]
frozen_rows=[]
for scope,smask in SCOPES:
    frame=test.loc[smask]; leaves=frozen_leaf[smask]
    for node,path in frozen_paths.items():
        for mode in WEIGHT_MODES:
            row,_=group_summary(frame,leaves==node,"leaf_"+str(node),scope,mode)
            keys={frozen_payload["preprocessor"]["features"][s["feature_index"]].get("key")
                  for s in path if frozen_payload["preprocessor"]["features"][s["feature_index"]]["kind"]=="numeric"}
            bad=frame[["x_"+k for k in keys]].isna().any(axis=1).to_numpy() if keys else np.zeros(len(frame),bool)
            row.update(leaf_id=node,conditions=describe_path(path,frozen_payload["preprocessor"]),
                       n_rule_path_imputed=int(bad[leaves==node].sum()),
                       n_unseen_type=int(frozen_flags["unseen_type"][smask][leaves==node].sum()),
                       n_missing_type=int(frame.loc[leaves==node,"type_value"].eq("__MISSING__").sum()))
            frozen_rows.append(row)
tables["frozen_rule_weighting_checks"]=pd.DataFrame(frozen_rows)
# Reproduce original point estimates independently; bootstrap seeds may differ.
old=pd.read_csv(C_DIR/"frozen_rule_cards.csv")
for _,row in tables["frozen_rule_weighting_checks"].query("weight == 'fire_source'").iterrows():
    old_scope="2023–2024" if row["scope"]=="2023_2024" else row["scope"]
    reference=old.loc[old["variant"].eq("condition_tree")&old["scope"].eq(old_scope)&old["leaf_id"].eq(row["leaf_id"])]
    require(len(reference)==1,"Missing 08C frozen reference row.")
    require(np.isclose(row["held_fraction"],reference.iloc[0]["fire_average_source_held"],rtol=0,atol=1e-12),"Frozen group rate changed.")
    require(np.isclose(row["within_fire_difference"],reference.iloc[0]["rule_minus_rest_held"],rtol=0,atol=1e-12),"Frozen contrast changed.")
phase_done.add("frozen")
print("All eight frozen rules and their observed rates reproduce 08C.")


## 5. Mapping representation and near-fire dozer diagnostics
The old selection raster has 25 m cells and signed distances capped at 251 m. Positive values mean raster-outside. The frozen cohort already passed those raster-based selection checks; their uniformly clean flags are not new validation.

The original polygon-to-line minimum, sampled boundary-distance minimum, and sampled boundary-distance median answer different questions. Zero polygon distance indicates intersection or touching; it does not identify how much fireline was crossed. The unsigned boundary distances do not establish vector-inside membership.

We compare uncapped raster minima with sampled boundary minima, retaining the capped and missing counts separately. A difference exceeding 25 m is a diagnostic flag, not proof of error. Alternative 25/50 m proximity definitions and intersection exclusions are stress tests, not a redefinition of NIFC outcomes.

The uncapped/small-discrepancy subset also excludes distant, capped observations. It is a changed-population sensitivity, not an isolated correction for map error.


In [ ]:
need("frozen")
RASTER_PATH=PROJECT_ROOT/"data/processed/fireline_dynamic_approach_v2/dynamic_approach_features_v2.parquet"
require(RASTER_PATH.is_file(),"Missing saved raster-distance supplement: "+str(RASTER_PATH))
input_stats[str(RASTER_PATH)]={"bytes":RASTER_PATH.stat().st_size,"modified_ns":RASTER_PATH.stat().st_mtime_ns}
raster_cols=["unit_uid","prediction_anchor_utc","t0_min_signed_distance_m","t0_p10_signed_distance_m",
 "t0_median_signed_distance_m","t0_inside_fraction","t0_within_50m_fraction","t0_outside_gt50m_fraction"]
raster=pd.read_parquet(RASTER_PATH,columns=raster_cols)
require(raster["unit_uid"].is_unique,"Duplicate raster supplement IDs.")
joined=test[["unit_uid","prediction_anchor_utc"]].merge(raster,on="unit_uid",how="left",validate="one_to_one",suffixes=("_test","_raster"))
require(joined["prediction_anchor_utc_test"].eq(joined["prediction_anchor_utc_raster"]).all(),"Raster anchors differ.")
for col in raster_cols[2:]: test[col]=joined[col].to_numpy()
test["raster_min_capped"]=test["t0_min_signed_distance_m"].abs().ge(251)
test["raster_minus_sampled_min"]=test["t0_min_signed_distance_m"]-test["exact_t0_exact_min_distance_m"]
test["uncapped_distance_disagreement_gt25m"]=(~test["raster_min_capped"])&test["raster_minus_sampled_min"].abs().gt(25)
mapping_rows=[]; mapping_outcomes=[]
for scope,smask in SCOPES:
    for group,gmask in [("all_test_units",np.ones(len(test),bool))]+[
        ("leaf_"+str(node),frozen_leaf==node) for node in frozen_paths]:
        part=test.loc[smask&gmask]; uncapped=part.loc[~part["raster_min_capped"]]
        row={"scope":scope,"group":group,"n_units":len(part),"n_fires":part["_irwin_norm"].nunique(),
          "polygon_distance_zero":int(part["x_distance"].eq(0).sum()),"raster_capped":int(part["raster_min_capped"].sum()),
          "raster_missing":int(part["t0_min_signed_distance_m"].isna().sum()),
          "sampled_p10_under25m":int(part["exact_t0_exact_p10_distance_m"].lt(25).sum()),
          "sampled_median_under25m":int(part["exact_t0_exact_median_distance_m"].lt(25).sum()),
          "sampled_median_under50m":int(part["exact_t0_exact_median_distance_m"].lt(50).sum()),
          "uncapped_disagreement_gt25m":int(part["uncapped_distance_disagreement_gt25m"].sum())}
        for col in ["x_distance","exact_t0_exact_min_distance_m","exact_t0_exact_p10_distance_m",
                    "exact_t0_exact_median_distance_m","actual_line_length_m"]:
            for q,label in [(0.1,"p10"),(.5,"median"),(.9,"p90")]:
                row[col+"_"+label]=part[col].quantile(q) if len(part) else np.nan
        for q,label in [(.5,"median"),(.9,"p90"),(.99,"p99")]:
            row["uncapped_raster_minus_sampled_min_"+label]=uncapped["raster_minus_sampled_min"].quantile(q) if len(uncapped) else np.nan
        mapping_rows.append(row)
    frame=test.loc[smask]
    dozer=frame["type_value"].eq("Completed Dozer Line").to_numpy()
    # Each comparator is the OTHER dozer units from the same eligible fires.
    dozer_frame=frame.loc[dozer]
    definitions={"polygon_intersects":dozer_frame["x_distance"].eq(0).to_numpy()}
    for cutoff in [25,50]:
        for col,label in [("x_distance","polygon_min"),("exact_t0_exact_min_distance_m","sampled_min"),
                          ("exact_t0_exact_p10_distance_m","sampled_p10"),("exact_t0_exact_median_distance_m","sampled_median")]:
            definitions[label+"_le_"+str(cutoff)]=dozer_frame[col].le(cutoff).to_numpy()
    for name,mask in definitions.items():
        for mode in ["fire_unit","fire_source"]:
            row,_=group_summary(dozer_frame,mask,name,scope,mode,"dozer_geometry_definitions")
            row["comparison_population"]="Dozer only; source weights recomputed within that population"
            mapping_outcomes.append(row)
    for label,valid in [
        ("all",np.ones(len(frame),bool)),
        ("strictly_positive_polygon_distance",frame["x_distance"].gt(0).to_numpy()),
        ("uncapped_raster_sample_difference_at_most25m",
         ((~frame["raster_min_capped"])&frame["raster_minus_sampled_min"].abs().le(25)).to_numpy())]:
        part=frame.loc[valid]; leaves=frozen_leaf[smask][valid]
        for node in frozen_paths:
            row,_=group_summary(part,leaves==node,"leaf_"+str(node),scope,"fire_source","geometry_exclusion")
            row["diagnostic_subset"]=label; mapping_outcomes.append(row)
tables["mapping_representation_audit"]=pd.DataFrame(mapping_rows)
tables["mapping_outcome_sensitivity"]=pd.DataFrame(mapping_outcomes)
flag_rows=[]
for col in ["operational_anchor_clean","prediction_t0_any_inside","exact_t0_cached_inside_fraction","exact_history_any_cached_inside"]:
    for value,count in units[col].value_counts(dropna=False).items():
        flag_rows.append({"field":col,"value":str(value),"units":int(count)})
tables["selection_flag_inventory"]=pd.DataFrame(flag_rows)
test[["unit_uid","row_id","_irwin_norm","source_record_key","Year","y","type_value","x_distance"]+
     diagnostic_columns+raster_cols[2:]+["raster_min_capped","raster_minus_sampled_min","uncapped_distance_disagreement_gt25m"]].assign(
     frozen_leaf=frozen_leaf).to_parquet(RUN_DIR/"08D_geometry_review_units.parquet",index=False)
phase_done.add("mapping")
display(tables["mapping_representation_audit"].query("scope == '2023_2024' and group in ['leaf_6','leaf_10']"))


## 6. Nearby cutoffs, without selecting a winner
The exact 08C humidity and distance cutoffs are included alongside fixed round-number alternatives. These grids were chosen after viewing 08C; they are exploratory stress tests, not new validated thresholds.

The grid uses observed humidity and distance and known fireline types. All settings are reported. The within-fire comparison is also repeated on the **same eligible fires across every cutoff in a family**, preventing changing fire composition from masquerading as cutoff stability. Sparse common-fire sets remain visible.

There are 13 cutoff settings. Keeping every setting visible reduces selective presentation but does not remove multiple-comparison uncertainty.


In [ ]:
need("mapping")
def original_cutoff(node,key,operator):
    values=[s["threshold"] for s in frozen_paths[node]
            if frozen_payload["preprocessor"]["features"][s["feature_index"]].get("key")==key and s["operator"]==operator]
    require(len(values)>0,"Missing reference threshold.")
    return min(values) if operator=="<=" else max(values)
RH_REF=original_cutoff(6,"humidity",">")
DIST_REF=original_cutoff(6,"distance","<=")
NEAR_REF=original_cutoff(10,"distance","<=")
NEAR_ANCESTOR=max(s["threshold"] for s in frozen_paths[10] if s["operator"]=="<=" and
 frozen_payload["preprocessor"]["features"][s["feature_index"]].get("key")=="distance")
GRID=[{"family":"humid_not_dozer","name":f"rh_{rh:.8g}__distance_{dist:.8g}","rh":rh,"distance":dist,
       "original_reference":rh==RH_REF and dist==DIST_REF}
      for rh in [30.,RH_REF,40.] for dist in [150.,DIST_REF,250.]]
GRID += [{"family":"near_dozer","name":f"dozer_distance_{dist:.8g}","rh":None,"distance":dist,
          "original_reference":dist==NEAR_REF} for dist in [5.,NEAR_REF,25.,50.]]
grid_rows=[];common_rows=[];grid_coverage=[]
known=set(frozen_payload["preprocessor"]["type_categories"])-{"__MISSING__","__UNSEEN__"}
for scope,smask in SCOPES:
    scope_frame=test.loc[smask]
    valid=(scope_frame["x_humidity"].notna()&scope_frame["x_distance"].notna()&scope_frame["type_value"].isin(known)).to_numpy()
    frame=scope_frame.loc[valid]
    grid_coverage.append({"scope":scope,"all_units":len(scope_frame),"complete_known_type_units":len(frame),
                         "excluded_units":int((~valid).sum()),"fires":frame["_irwin_norm"].nunique()})
    # Match the original float32 representation, comparing against double thresholds.
    rh=frame["x_humidity"].to_numpy(dtype=np.float32).astype(np.float64)
    dist=frame["x_distance"].to_numpy(dtype=np.float32).astype(np.float64)
    dozer=frame["type_value"].eq("Completed Dozer Line").to_numpy()
    for mode in ["fire_unit","fire_source"]:
        contrasts={}
        for setting in GRID:
            if setting["family"]=="humid_not_dozer":
                mask=(~dozer)&(rh>setting["rh"])&(dist<=setting["distance"])
            else: mask=dozer&(dist<=NEAR_ANCESTOR)&(dist<=setting["distance"])
            if setting["original_reference"]:
                reference_node=6 if setting["family"]=="humid_not_dozer" else 10
                require(np.array_equal(mask,frozen_leaf[smask][valid]==reference_node),"Grid reference path does not reproduce frozen group.")
            row,diffs=group_summary(frame,mask,setting["name"],scope,mode,setting["family"])
            row.update(humidity_cutoff=setting["rh"],distance_cutoff=setting["distance"],original_reference=setting["original_reference"])
            grid_rows.append(row); contrasts[setting["name"]]=diffs
        for family in ["humid_not_dozer","near_dozer"]:
            settings=[s for s in GRID if s["family"]==family]
            common=set(contrasts[settings[0]["name"]].index)
            for setting in settings[1:]: common &= set(contrasts[setting["name"]].index)
            ids=sorted(common)
            for setting in settings:
                values=contrasts[setting["name"]].reindex(ids)
                point,lo,hi=stat_from_parts(pd.DataFrame({"num":values,"den":1.0}),"grid_common|"+scope+"|"+family)
                common_rows.append({"scope":scope,"weight":mode,"family":family,"group":setting["name"],
                  "original_reference":setting["original_reference"],"common_fires":len(ids),
                  "within_fire_difference":point,"ci_lower":lo,"ci_upper":hi})
tables["cutoff_sensitivity"]=pd.DataFrame(grid_rows)
tables["cutoff_common_fire_sensitivity"]=pd.DataFrame(common_rows)
tables["cutoff_population_coverage"]=pd.DataFrame(grid_coverage)
phase_done.add("cutoffs")
display(tables["cutoff_common_fire_sensitivity"].query("scope == '2023_2024' and weight == 'fire_source'"))


## 7. Nine fixed tree-construction sensitivities
All trees train on 2020–2022 and remain unchanged for both 2023 and 2024. We cross depths 2, 3 and 4 with fire/unit, fire/source, and source-only training weights. Training support requirements are unchanged.

Changing weights also changes training-weighted median imputation; this is a sensitivity of the whole fitting procedure, not just one coefficient. All nine are retained. No test outcome chooses depth, predictors, preprocessing or pruning.

The depth-3/fire-source tree must reproduce the original 08C memberships and probabilities exactly. Group IDs from different trees are not interchangeable. We report how the original groups are distributed across new groups, rather than calling the highest-overlap leaf a validated replacement.


In [ ]:
need("cutoffs")
construction_rows=[];construction_rules=[];construction_groups=[];construction_overlap=[];construction_predictions=[]
train=units.loc[units["Year"].le(2022)].copy().reset_index(drop=True)
assert_split(train,test,chronological=True)
priors={mode:float(np.average(train["y"],weights=training_weights(train,mode))) for mode in WEIGHT_MODES}
plan={"build_id":BUILD_ID,"input_run":C_RUN_NAME,"weight_modes":WEIGHT_LABELS,"tree_specs":TREE_SPECS,
 "cutoff_grid":GRID,"frozen_training_years":[2020,2021,2022],"test_years":[2023,2024],
 "bootstrap_replicates":BOOTSTRAP_REPLICATES,"is_smoke_test":IS_SMOKE_TEST,"seed":RANDOM_SEED,
 "support":{"train_units":MIN_LEAF_UNITS,"train_sources":MIN_LEAF_PARENTS,"train_fires":MIN_LEAF_FIRES,
            "test_units":MIN_TEST_UNITS,"test_sources":MIN_TEST_PARENTS,"test_fires":MIN_TEST_FIRES},
 "interpretation":"All fixed settings are exploratory; no winner selected using test results.",
 "distance_note":"Polygon-to-line minimum differs from sampled unsigned boundary distances and capped raster distances."}
write_json(RUN_DIR/"08D_fixed_run_plan.json",plan)
for setting in TREE_SPECS:
    started=time.perf_counter()
    spec={"name":setting["name"],"keys":ENVIRONMENT_KEYS,"use_type":True,"weight_mode":setting["weight_mode"],"population":"all"}
    bundle=fit_rule_tree(train,spec,setting["name"],max_depth=setting["depth"])
    p,leaf,flags=predict_rule_tree(bundle,test)
    fingerprint=bundle["fingerprint"]
    if setting["name"]=="depth_3__fire_source":
        require(np.array_equal(leaf,frozen_leaf) and np.array_equal(p,frozen_p),"Reference tree did not reproduce 08C.")
    folder=MODEL_DIR/setting["name"]; folder.mkdir(exist_ok=False)
    write_json(folder/"model.json",{**bundle["payload"],"fingerprint":fingerprint})
    pred=test[["row_id","unit_uid","Year"]].copy()
    pred["model"]=setting["name"]; pred["leaf_id"]=leaf;pred["p_held"]=p
    pred["any_numeric_missing"]=flags["any_numeric_missing"];pred["unseen_type"]=flags["unseen_type"]
    pred.to_parquet(folder/"test_predictions.parquet",index=False);construction_predictions.append(pred)
    for node,path in bundle["paths"].items():
        construction_rules.append({"model":setting["name"],"leaf_id":node,"conditions":describe_path(path,bundle["prep"]),
          "fitted_held_probability":bundle["leaf_probs"][node],**bundle["payload"]["leaf_training_support"][node]})
    for scope,smask in SCOPES:
        frame=test.loc[smask]; ll=leaf[smask]
        predictions={setting["name"]:p[smask],**{"prior_"+mode:np.full(len(frame),priors[mode]) for mode in WEIGHT_MODES}}
        construction_rows.extend(error_comparisons(frame,predictions,[setting["name"]],
          ["prior_"+m for m in WEIGHT_MODES],scope,"frozen_2020_2022"))
        for node in bundle["paths"]:
            row,_=group_summary(frame,ll==node,"leaf_"+str(node),scope,"fire_source","construction")
            row.update(model=setting["name"],leaf_id=node,
                       n_any_numeric_missing=int(flags["any_numeric_missing"][smask][ll==node].sum()),
                       n_unseen_type=int(flags["unseen_type"][smask][ll==node].sum()))
            construction_groups.append(row)
        w=training_weights(frame,"fire_source")
        for original in frozen_paths:
            a=frozen_leaf[smask]==original
            for node in bundle["paths"]:
                b=ll==node; intersection=a&b; union=a|b
                construction_overlap.append({"model":setting["name"],"scope":scope,"original_leaf":original,"new_leaf":node,
                  "intersection_units":int(intersection.sum()),"original_units":int(a.sum()),"new_units":int(b.sum()),
                  "weighted_jaccard":float(w[intersection].sum()/w[union].sum()) if union.any() else np.nan,
                  "original_weight_fraction_in_new":float(w[intersection].sum()/w[a].sum()) if a.any() else np.nan})
    require(model_fingerprint(bundle["payload"])==fingerprint,"Fitted rule changed during evaluation.")
    pd.DataFrame(construction_rows).to_csv(RUN_DIR/"construction_performance_progress.csv",index=False)
    print(setting["name"],len(bundle["paths"]),"groups;",round(time.perf_counter()-started,1),"seconds",flush=True)
tables["construction_performance"]=pd.DataFrame(construction_rows)
tables["construction_rule_paths"]=pd.DataFrame(construction_rules)
tables["construction_group_outcomes"]=pd.DataFrame(construction_groups)
tables["construction_group_overlap"]=pd.DataFrame(construction_overlap)
pd.concat(construction_predictions,ignore_index=True).to_parquet(RUN_DIR/"08D_construction_predictions.parquet",index=False)
phase_done.add("construction")


## 8. Figures and review report
The baseline figure shows a model-minus-baseline difference: negative values favor the model. Every row uses one consistent evaluation weighting. The distance and cutoff figures summarize diagnostics, not operational thresholds. CSV tables include the full comparisons and sample support.


In [ ]:
need("construction")
plt.rcParams.update({"font.size":10,"axes.spines.top":False,"axes.spines.right":False})
figure_files=[]
def save_figure(fig,name):
    if IS_SMOKE_TEST: fig.text(.99,.005,f"SOFTWARE CHECK: {BOOTSTRAP_REPLICATES} resamples",ha="right",color="#b65a00")
    for extension in ["png","pdf"]:
        path=FIG_DIR/(name+"."+extension)
        fig.savefig(path,dpi=180,bbox_inches="tight")
        figure_files.append(str(path.relative_to(RUN_DIR)))
    display(fig);plt.close(fig)

d=tables["baseline_error_comparisons"].query("scheme == 'forward_year' and scope == 'all_test_years' and metric == 'brier' and model == 'LightGBM'")
matrix=d.pivot(index="evaluation_weight",columns="baseline_weight",values="difference").reindex(index=WEIGHT_MODES,columns=WEIGHT_MODES)
fig,ax=plt.subplots(figsize=(9,6))
limit=max(float(np.abs(matrix).max().max()),1e-4)
im=ax.imshow(matrix,cmap="RdBu_r",vmin=-limit,vmax=limit)
short=["Unit","Source","Fire/unit","Fire/source"]
ax.set_xticks(range(4),short);ax.set_yticks(range(4),short)
for i in range(4):
    for j in range(4):
        value=matrix.iloc[i,j]
        ax.text(j,i,f"{value:+.4f}",ha="center",va="center",color="white" if abs(value)>.55*limit else "black")
        if i==j: ax.add_patch(plt.Rectangle((j-.48,i-.48),.96,.96,fill=False,edgecolor="black",linewidth=2))
ax.set_xlabel("Weighting used for the training-only Held fraction")
ax.set_ylabel("Weighting used to evaluate probability error")
ax.set_title("LightGBM forward-year comparison depends on the baseline\nBrier difference: model minus constant; diagonal boxes match weighting")
fig.colorbar(im,ax=ax,label="Negative favors LightGBM")
fig.tight_layout();save_figure(fig,"08D_baseline_weighting_matrix")

near=test.loc[frozen_leaf==10]
fig,axes=plt.subplots(1,3,figsize=(13,4.5),sharey=True)
dist_specs=[("x_distance","Closest polygon-to-line distance"),
 ("exact_t0_exact_median_distance_m","Median sampled boundary distance"),
 ("t0_min_signed_distance_m","Minimum signed raster distance")]
for ax,(col,label) in zip(axes,dist_specs):
    values=np.sort(near[col].dropna().to_numpy())
    if len(values): ax.step(values,np.arange(1,len(values)+1)/len(values),where="post",color="#0072B2",linewidth=2)
    ax.set_title(label);ax.set_xlabel("Distance (m)")
    ax.set_xlim(0,100);ax.set_ylim(0,1);ax.yaxis.set_major_formatter(PercentFormatter(1))
axes[0].set_ylabel("Share of fireline units at or below distance")
fig.suptitle("Original near-fire dozer group: different distance summaries\nView limited to 0–100 m; full quantiles and capped counts are in the audit table")
fig.tight_layout();save_figure(fig,"08D_near_dozer_distance_definitions")

d=tables["cutoff_common_fire_sensitivity"].query("scope == '2023_2024' and weight == 'fire_source'").copy()
fig,axes=plt.subplots(1,2,figsize=(13,6),sharex=True)
for ax,family in zip(axes,["humid_not_dozer","near_dozer"]):
    part=d.loc[d.family.eq(family)].reset_index(drop=True)
    for i,row in part.iterrows():
        color="#D55E00" if row["original_reference"] else "#0072B2"
        if np.isfinite(row["within_fire_difference"]):
            ax.hlines(i,row["ci_lower"]*100,row["ci_upper"]*100,color=color)
            ax.plot(row["within_fire_difference"]*100,i,"o",color=color)
    ax.axvline(0,color="gray",ls="--")
    labels=[]
    for group_name in part["group"]:
        setting=next(s for s in GRID if s["name"]==group_name)
        labels.append((f"RH > {setting['rh']:.1f}%; " if setting["rh"] is not None else "")+f"distance <= {setting['distance']:.1f} m")
    ax.set_yticks(range(len(part)),labels)
    ax.invert_yaxis();ax.set_xlabel("Held difference within the same fires (percentage points)")
    n=int(part["common_fires"].iloc[0]) if len(part) else 0
    ax.set_title(family.replace("_"," ")+f"\n{n} common eligible fires; orange = original cutoff")
finite_lo=d["ci_lower"].dropna();finite_hi=d["ci_upper"].dropna()
common_lo=min(0.,float(finite_lo.min())*100) if len(finite_lo) else 0.
common_hi=max(0.,float(finite_hi.max())*100) if len(finite_hi) else 0.
for ax in axes: ax.set_xlim(common_lo-2,common_hi+2)
fig.suptitle("All fixed cutoff checks, 2023–2024\nPointwise 95% intervals; no cutoff selected as a winner")
fig.tight_layout();save_figure(fig,"08D_cutoff_stability")

def md_table(frame,cols):
    head="| "+" | ".join(cols)+" |\n| "+" | ".join(["---"]*len(cols))+" |\n"
    return head+"\n".join("| "+" | ".join(str(v) for v in row)+" |" for row in frame[cols].itertuples(index=False,name=None))
summary=tables["baseline_error_comparisons"].query("scope == 'all_test_years' and metric == 'brier' and matched_weighting").copy()
for c in ["model_error","baseline_error","difference","ci_lower","ci_upper"]:summary[c]=summary[c].map(lambda x:f"{x:.4f}")
report=f"""# 08D — Baseline and condition robustness
Run {RUN_ID}. Bootstrap resamples: {BOOTSTRAP_REPLICATES:,}. Software-check run: {IS_SMOKE_TEST}.

This follow-up was designed after 08C. It evaluates sensitivity, not independent confirmation.

## Matched baseline and evaluation weights
Negative differences favor the model. The full table additionally crosses all baseline/scoring choices.
{md_table(summary,["scheme","model","evaluation_weight","model_error","baseline_error","difference","ci_lower","ci_upper"])}

## How to review the condition findings
- frozen_rule_weighting_checks.csv: every original group, both test years and combined, four averaging choices.
- mapping_representation_audit.csv: polygon intersections, sampled boundary distances, capped raster distances and uncapped differences.
- mapping_outcome_sensitivity.csv: alternate distance definitions and diagnostic exclusions, with changed populations clearly labeled.
- cutoff_sensitivity.csv and cutoff_common_fire_sensitivity.csv: every fixed nearby cutoff; the latter holds eligible fires constant across a family.
- construction_performance.csv: all nine depth/weight procedures; no best test-year configuration is selected.
- construction_rule_paths.csv: rounded readable conditions; models/<model>/model.json: exact thresholds and preprocessing. construction_group_overlap.csv shows correspondence with the original eight groups.

## Required interpretation
The geometric predictor measures the closest polygon-to-line distance, not whole-fireline proximity. Sampled unsigned boundary distances cannot determine vector-inside status. The old raster caps distance at 251 m. A grid/vector difference is not proof of a mapping error.

Group percentages differ from raw unit Held fractions when records or fires are weighted. Source shares are allocated before group membership is applied. Common-fire contrasts require at least five source records on each side.

All intervals are pointwise and conditional on fitted predictions. Multiple checks, prior archive exploration, source-label-dependent selection, and confounding prevent causal or operational interpretation. Sparse and degenerate groups remain in the tables.

The nine trees also vary training-weighted imputation, so weighting comparisons concern the fitting procedure as a whole. The existing LightGBM model was not trained with all four weight choices.

## What this notebook does not change
No source labels, fireline units, assigned times, existing models, earlier outputs, or manuscript files are changed. No width/length construction recommendation is inferred.
"""
(RUN_DIR/"08D_FINDINGS_REPORT.md").write_text(report,encoding="utf-8")
phase_done.add("figures")


## 9. Save and verify preservation
The manifest and checks are written only after every preceding phase succeeds. Existing inputs must have unchanged sizes and modification times. All key comparison tables and unit-level prediction/diagnostic files are retained.


In [ ]:
need("figures")
require(len(units)==304143 and digest(units,identity,["unit_uid"])==identity_hash,"Frozen identity or outcomes changed.")
require(saved_notebook_source_hash()==NOTEBOOK_SOURCE_HASH_AT_START,"Notebook source changed during execution; save a fixed version and restart.")
for path,stats in input_stats.items():
    p=Path(path)
    require(p.stat().st_size==stats["bytes"] and p.stat().st_mtime_ns==stats["modified_ns"],"Existing input changed: "+path)
for name,table in tables.items(): table.to_csv(RUN_DIR/(name+".csv"),index=False,encoding="utf-8-sig")
write_json(RUN_DIR/"08D_quality_checks.json",{
 "is_smoke_test":IS_SMOKE_TEST,"frozen_ids_labels_and_times_preserved":True,
 "baseline_training_fire_separation":True,"four_training_priors_recomputed":True,
 "original_unit_prior_and_08C_prior_reproduced":True,"all_evaluation_populations_matched":True,
 "frozen_rules_and_group_rates_reproduced":True,"reference_depth3_fire_source_tree_reproduced":True,
 "all_nine_construction_variants_completed":len(TREE_SPECS)==9,
 "original_input_stats_unchanged":True,"no_test_selected_winner":True})
write_json(RUN_DIR/"08D_run_manifest.json",{**plan,"run_id":RUN_ID,
 "completed_utc":datetime.now(timezone.utc).isoformat(),"input_files":input_stats,
 "notebook_source_hash":NOTEBOOK_SOURCE_HASH_AT_START,"identity_hash":identity_hash,
 "python":sys.version,"package_versions":{n:package_version(n) for n in required},"figures":figure_files,
 "table_rows":{name:len(table) for name,table in tables.items()}})
inventory=[{"path":str(p.relative_to(RUN_DIR)),"bytes":p.stat().st_size} for p in sorted(RUN_DIR.rglob("*")) if p.is_file()]
write_json(RUN_DIR/"08D_output_inventory.json",inventory)
phase_done.add("complete")
print("08D COMPLETE:",RUN_DIR)
print("Start with 08D_FINDINGS_REPORT.md and baseline_error_comparisons.csv.")


## After the run
Review the baseline matrix before revising broad claims about improvement over historical prediction. For condition findings, examine uncertainty, same-fire contrasts, sample support, map representations and stability across years together.

Do not select the most favorable cutoff or tree and call the same 2023–2024 observations a new independent test. Further confirmatory work requires separate fires or a prospectively fixed evaluation.

Save the notebook in Antigravity after execution so its displayed execution record matches the completed result folder.
